## Preprocessing

Input: `data/processed/train.parquet`, `data/processed/test.parquet` (saved từ `EDA.ipynb`)

Output: `data/processed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

Các bước:
1. Fill NaN trong dynamic features (VLE + assessment)
2. Impute `imd_band = '?'` theo phân phối từ train
3. Encode categorical features
4. Lưu feature matrix

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import DB_PATH, SNAPSHOTS, STATIC_FEATURES, RANDOM_SEED

import numpy as np
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_parquet('../data/processed/train.parquet')
test  = pd.read_parquet('../data/processed/test.parquet')

# Cần assessment để tính lại num_due trong fill_in_assessment
conn = sqlite3.connect(DB_PATH)
assessment = pd.read_sql("SELECT * FROM assessments", conn)
conn.close()

print('train:', train.shape)
print('test: ', test.shape)

train: (142384, 22)
test:  (35910, 22)


---

### 1. Fill NaN — Dynamic features

In [3]:
from src.features.build_features import (
    fill_in_weekly_clicks, fill_in_assessment,
    fit_imd_distributions, transform_imd_imputation,
    DISAB_MAP, EDU_ORDER, IMD_ORDER,
)

train = fill_in_weekly_clicks(train)
train = fill_in_assessment(train, assessment)

test = fill_in_weekly_clicks(test)
test = fill_in_assessment(test, assessment)

In [4]:
# avg_score_filled: -1 sentinel (ngoài range [0,100] — phân biệt với điểm 0 thực)
# avg_days_early_filled: 0 (no_submission_despite_due đã capture trường hợp không nộp)
for df in [train, test]:
    df['avg_days_early_filled'] = df['avg_days_early'].fillna(0)
    df['avg_score_filled']      = df['avg_score'].fillna(-1)

dynamic_features = [
    'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled',
    'num_due', 'num_submitted_filled', 'avg_score_filled',
    'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due',
]

null_counts = pd.DataFrame({
    'train': train[dynamic_features].isna().sum(),
    'test':  test[dynamic_features].isna().sum(),
})
print(null_counts[null_counts.any(axis=1)])

Empty DataFrame
Columns: [train, test]
Index: []


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 142384 entries, 0 to 142383
Data columns (total 31 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_student                 142384 non-null  int64  
 1   code_module                142384 non-null  object 
 2   code_presentation          142384 non-null  object 
 3   gender                     142384 non-null  object 
 4   region                     142384 non-null  object 
 5   age_band                   142384 non-null  object 
 6   imd_band                   142384 non-null  object 
 7   highest_education          142384 non-null  object 
 8   disability                 142384 non-null  object 
 9   num_of_prev_attempts       142384 non-null  int64  
 10  studied_credits            142384 non-null  int64  
 11  date_registration          142384 non-null  object 
 12  date_unregistration        12920 non-null   float64
 13  total_clicks               13

---

### 2. Impute `imd_band = '?'`

Tạo biến chỉ báo `imd_missing`, sau đó impute theo phân phối riêng của từng `region` (ước lượng trên tập train).

In [6]:
# Tạo indicator trước khi impute
for df in [train, test]:
    df['imd_missing'] = (df['imd_band'] == '?').astype(int)

print(f"imd_missing — train: {train['imd_missing'].sum():,} | test: {test['imd_missing'].sum():,}")
print()

# Impute theo phân phối của từng region (fit trên train, áp dụng cho cả test)
imd_distributions = fit_imd_distributions(train)
rng = np.random.default_rng(RANDOM_SEED)
train = transform_imd_imputation(train, imd_distributions, rng)
test  = transform_imd_imputation(test,  imd_distributions, rng)
n_remain_train = (train['imd_band_filled'] == '?').sum()
n_remain_test  = (test['imd_band_filled'] == '?').sum()
print(f"Sau impute — imd_band_filled '?' còn lại: train={n_remain_train} | test={n_remain_test}")


imd_missing — train: 5,986 | test: 1,497



Sau impute — imd_band_filled '?' còn lại: train=0 | test=0


---

### 3. Encode categorical features

| Feature | Kiểu | Lý do |
|---|---|---|
| `disability` | Binary | Y→1, N→0 |
| `highest_education` | Ordinal | Có thứ tự theo bậc học |
| `imd_band_filled` | Ordinal | Có thứ tự theo mức độ nghèo |

> `gender`, `region`, `age_band`, `prediction_point` bị loại khỏi feature matrix (xem `DROP_COLS`). Encode vẫn chạy để giữ train/test sạch, nhưng các cột đó không đưa vào X.

In [7]:
# Maps imported từ build_features: DISAB_MAP, EDU_ORDER, IMD_ORDER
for df in [train, test]:
    df['disability']        = df['disability'].map(DISAB_MAP)
    df['highest_education'] = df['highest_education'].map(EDU_ORDER)
    df['imd_band_filled']   = df['imd_band_filled'].map(IMD_ORDER)

# Label encode target — fit trên train
labels = sorted(train['final_result'].unique())
LABEL_MAP = {l: i for i, l in enumerate(labels)}
print('Label map:', LABEL_MAP)

Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}


---

### 4. Build feature matrices

In [8]:
DROP_COLS = {'gender', 'region', 'age_band', 'imd_band'}
# prediction_point không phải feature — không đưa vào X, lưu riêng để modeling tách theo mốc
FEATURE_COLS = (
    [c for c in STATIC_FEATURES + dynamic_features if c not in DROP_COLS]
    + ['imd_band_filled', 'imd_missing']
)

X_train = train[FEATURE_COLS].copy()
X_test  = test[FEATURE_COLS].copy()

y_train = train['final_result'].map(LABEL_MAP)
y_test  = test['final_result'].map(LABEL_MAP)

print('X_train:', X_train.shape, '| NaN:', X_train.isna().sum().sum())
print('X_test: ', X_test.shape,  '| NaN:', X_test.isna().sum().sum())
print('y_train:\n', y_train.value_counts().sort_index())
print('y_test:\n',  y_test.value_counts().sort_index())
print('\nFeatures:', FEATURE_COLS)

X_train: (142384, 15) | NaN: 0
X_test:  (35910, 15) | NaN: 0
y_train:
 final_result
0    40333
1    88536
2    13515
Name: count, dtype: int64
y_test:
 final_result
0    10192
1    22432
2     3286
Name: count, dtype: int64

Features: ['highest_education', 'disability', 'num_of_prev_attempts', 'studied_credits', 'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled', 'num_due', 'num_submitted_filled', 'avg_score_filled', 'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due', 'imd_band_filled', 'imd_missing']


In [9]:
out_dir = Path('../data/processed')
out_dir.mkdir(exist_ok=True)

X_train.to_parquet(out_dir / 'X_train.parquet', index=False)
X_test.to_parquet(out_dir  / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(out_dir / 'y_train.parquet', index=False)
y_test.to_frame().to_parquet(out_dir  / 'y_test.parquet',  index=False)

# Lưu prediction_point riêng để modeling dùng tách dữ liệu theo từng mốc
train[['prediction_point']].to_parquet(out_dir / 'train_meta.parquet', index=False)
test[['prediction_point']].to_parquet(out_dir  / 'test_meta.parquet',  index=False)

print('Saved to data/processed/')
print('Label map:', LABEL_MAP)
print('Feature cols:', FEATURE_COLS)

Saved to data/processed/
Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}
Feature cols: ['highest_education', 'disability', 'num_of_prev_attempts', 'studied_credits', 'total_clicks_filled', 'active_weeks_filled', 'avg_weekly_clicks_filled', 'num_due', 'num_submitted_filled', 'avg_score_filled', 'num_failed_filled', 'avg_days_early_filled', 'no_submission_despite_due', 'imd_band_filled', 'imd_missing']
